# Практика. Векторные вычисления

In [2]:
import numpy as np
import pandas as pd
import time

Настоящее занятие направлено на решение ключевой проблемы оптимизации вычислительных процессов: перехода от последовательного (итерационного) к параллельному (векторному) стилю мышления. В рамках этого мы:

**Устраним императивный подход:** убедимся, что использование циклов для операций над элементами данных является архаичным и неэффективным методом, и изучим синтаксис и семантику их векторных альтернатив.

**Освоим аппарат условной селекции:** исследуем булевы маски как основной инструмент декларативного описания условий для фильтрации, уделив особое внимание комбинированию простых предикатов в сложные логические выражения.

>#### Задание 1
Загрузите датасет retail из папки data. Просмотрите информацию о данных при помощи метода `info()`

In [8]:
df = pd.read_csv('data/retail.csv')
display(df)
df.info()

,transaction_id,product_category,quantity,is_member,hour_of_day,unit_price
0,T047283,Clothing,20,1,15,439.93
1,T093646,Electronics,10,0,16,1176.39
2,T010669,Groceries,18,1,21,342.14
3,T096743,Electronics,5,0,10,3771.35
4,T043460,Clothing,11,0,20,1415.86
...,...,...,...,...,...,...
99995,T085954,Home,7,1,9,1256.16
99996,T087063,Home,11,0,17,889.21
99997,T046509,Books,6,0,15,527.84
99998,T096124,Home,12,1,19,1190.78


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   transaction_id    100000 non-null  object 
 1   product_category  100000 non-null  object 
 2   quantity          100000 non-null  int64  
 3   is_member         100000 non-null  int64  
 4   hour_of_day       100000 non-null  int64  
 5   unit_price        100000 non-null  float64
dtypes: float64(1), int64(3), object(2)
memory usage: 4.6+ MB


Данные одержат сведения о транзакциях в розничном магазине:
- transaction_id — уникальный идентификатор транзакции (строка, например 'T000001'),
- product_category — категория товара: 'Electronics', 'Clothing', 'Groceries', 'Home', 'Books',
- quantity — количество товара в транзакции (целое число, от 1 до 20),
- unit_price — цена за единицу товара в рублях (число с плавающей точкой, от 50 до 5000),
- is_member — флаг членства в программе лояльности (0 — нет, 1 — да),
- hour_of_day — час дня, когда совершена покупка (целое число, от 9 до 21 — рабочие часы магазина).

>#### Задание 2
>Расчитайте суммарную выручку:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [9]:
start_time_for_loop = time.time()

total = 0

for i in range(len(df)):
    total = total + df['unit_price'][i]*df['quantity'][i]
    
time_for_loop = time.time() - start_time_for_loop

print('total =', round(total, 2))
print('time_for_loop =', time_for_loop)

total = 1317235305.42
time_for_loop = 0.5032289028167725


In [10]:
start_time_vector = time.time()

total = (df['unit_price']*df['quantity']).sum()

time_vector = time.time() - start_time_vector

print('total =', round(total, 2))
print('time_vector =', time_vector)

total = 1317235305.42
time_vector = 0.0019996166229248047


In [11]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 252


>#### Задание 3
Будем считать, что транзакция относится к категории Premium, если одновременно выполняются два условия: 1) сумма транзакции превышает 99000, 2) приобретаются товары из категории Electronics или Home. Найдите премиальные транзакции:
>1. итерационно, при помощи циклов,
>2. при помощи построения сложной булевой маски.
>
>Измерьте время исполнения, сравните скорость

In [12]:
start_time_for_loop = time.time()

premium = []

for i in range(len(df)):
    A = df['unit_price'][i]*df['quantity'][i] > 99000
    B = df['product_category'][i] == 'Electronics' or df['product_category'][i] == 'Home'
    if A and B:
        premium.append(df['transaction_id'][i])

time_for_loop = time.time() - start_time_for_loop

print('premium =', premium)
print('time_for_loop =', time_for_loop)

premium = ['T076983', 'T006588', 'T048118', 'T057927', 'T089571', 'T021749', 'T076463', 'T081658', 'T062328', 'T000959', 'T010936', 'T082661', 'T014331']
time_for_loop = 0.9166967868804932


In [13]:
start_time_vector = time.time()

transaction_id_array   = df['transaction_id'].values
unit_price_array       = df['unit_price'].values
quantity_array         = df['quantity'].values
product_category_array = df['product_category'].values

A = unit_price_array*quantity_array > 99000
B = (product_category_array == 'Electronics') | (product_category_array == 'Home')

mask = A & B

premium = transaction_id_array[mask]

time_vector = time.time() - start_time_vector

print('premium =', premium)
print('time_vector =', time_vector)

premium = ['T076983' 'T006588' 'T048118' 'T057927' 'T089571' 'T021749' 'T076463'
 'T081658' 'T062328' 'T000959' 'T010936' 'T082661' 'T014331']
time_vector = 0.0035364627838134766


In [14]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 259


>#### Задание 4
Найдите среднюю сумму транзакции для участников программы лояльности и для всех остальных покупателей. Выполните это задание:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [ ]:
start_time_for_loop = time.time()

Sum_member = 0
count_member = 0
Sum_not_member = 0
count_not_member = 0

for i in range(len(df)):
    if df['is_member'][i] == 1:
        Sum_member = Sum_member + df['unit_price'][i]*df['quantity'][i]
        count_member = count_member + 1
    else:
        Sum_not_member = Sum_not_member + df['unit_price'][i]*df['quantity'][i]
        count_not_member = count_not_member + 1

sum_member_average = round(Sum_member/count_member, 2)
sum_not_member_average = round(Sum_not_member/count_not_member, 2)

time_for_loop = time.time() - start_time_for_loop

print('sum_member_average     =', sum_member_average)
print('sum_not_member_average =', sum_not_member_average)
print('time_for_loop =', time_for_loop)

In [ ]:
start_time_vector = time.time()

total =(df['unit_price']*df['quantity']).values
member_mask = (df['is_member'] == 1).values

sum_member_average = round(total[member_mask].mean(), 2)
sum_not_member_average = round(total[~member_mask].mean(), 2)

time_vector = time.time() - start_time_vector

print('sum_member_average     =', sum_member_average)
print('sum_not_member_average =', sum_not_member_average)
print('time_vector =', time_vector)

In [ ]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

>#### Задание 5
Определите, в какие часы участники программы лояльности совершают больше всего покупок (по количеству транзакций). Выполните это задание:
>1. итерационно, при помощи циклов,
>2. при помощи векторных операций.
>
>Измерьте время исполнения, сравните скорость.

In [5]:
start_time_for_loop = time.time()

count_list = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

for hour in range(9, 22):
    for i in range(len(df)):
        if (df['is_member'][i] == 1) and (df['hour_of_day'][i] == hour):
            count_list[hour - 9] = count_list[hour - 9] + 1
            
max_count = 0
for count in count_list:
    if count > max_count:
        max_count = count

hour_max_transaction = count_list.index(max_count) + 9

time_for_loop = time.time() - start_time_for_loop

print('hour_max_transaction =', hour_max_transaction)
print('time_for_loop =', time_for_loop)

hour_max_transaction = 16
time_for_loop = 4.165781497955322


In [6]:
start_time_vector = time.time()

mask = df['is_member'] == 1
hour_max_transaction = df['hour_of_day'][mask].value_counts().index[0]

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('hour_max_transaction =', hour_max_transaction)
print('time_vector =', time_vector)

hour_max_transaction = 16
time_vector = 0.006003379821777344


In [7]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

speed_up = 694


# Домашнее задание

>#### Задание
Прочитайте файл server_logs.csv из папки data.

In [ ]:
df = pd.read_csv('data/server_logs.csv')
df

Данные содержат записи о HTTP-запросах к API веб-сервиса за сутки. Структура данных:
- request_id — уникальный идентификатор запроса (строка, например 'REQ-000001'),
- endpoint — конечная точка API: '/users', '/orders', '/products', '/auth/login', '/auth/logout',
- response_time_ms — время обработки запроса сервером в миллисекундах (целое число, от 20 до 5000),
- status_code — HTTP-статус ответа (целое число, например 200, 201, 400, 401, 404, 500),
- client_type — тип клиента: 'browser', 'mobile_app', 'script', 'other',
- request_size_kb — размер тела запроса в килобайтах (число с плавающей точкой, от 0.1 до 1024.0).

>#### Задание
>1. Рассчитайте нагрузку и эффективность:
>- добавьте новый столбец is_slow (булев), который равен True, если время отклика превышает 1000 мс (1 секунду),
>- добавьте новый столбец is_large_request (булев), который равен True, если размер запроса превышает 500 КБ.
>2. Выделите критические запросы по двум независимым критериям:
>- критерий A — запрос завершился с ошибкой клиента или сервера (status_code >= 400),
>- критерий B — запрос является одновременно медленным (is_slow == True) и большим (is_large_request == True),
>- запрос считается критическим, если выполняется хотя бы один из этих критериев.
>3. Проанализируйте поведение разных типов клиентов:
>- Найдите среднее время отклика отдельно для запросов от 'mobile_app' и для всех остальных типов клиентов.
>- Определите, есть ли статусы ошибок (status_code >= 400) среди запросов, отправленных из 'browser'.

Выполните все задания сналача в классической манере — при помощи циклов, а затем — в векторной манере. Сравните время исполнения алгоритмов.

#### Задание 1

In [ ]:
df = pd.read_csv('data/server_logs.csv')
start_time_for_loop = time.time()

is_slow = []
is_large = []

for i in range(len(df)):
    if df['response_time_ms'][i] > 1000:
        is_slow.append(True)
    else:
        is_slow.append(False)
        
for i in range(len(df)):        
    if df['request_size_kb'][i] > 500:
        is_large.append(True)
    else:
        is_large.append(False)
        
df['is_slow'] = is_slow
df['is_large'] = is_large

time_for_loop = time.time() - start_time_for_loop

display(df)
print('time_for_loop =', time_for_loop)

In [ ]:
df = pd.read_csv('data/server_logs.csv')

start_time_vector = time.time()

mask_time = df['response_time_ms'] > 1000
df['is_slow'] = mask_time
mask_size = df['request_size_kb'] > 500
df['is_large'] = mask_size

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005
    
display(df)
print('time_vector =', time_vector)

In [ ]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

#### Задание 2

In [ ]:
start_time_for_loop = time.time()

request_cr = []

for i in range(len(df)):
    A = df['status_code'][i] >= 400
    B = df['is_slow'][i] and df['is_large'][i]
    if A or B:
        request_cr.append(df['request_id'][i])

time_for_loop = time.time() - start_time_for_loop

print('request_cr[:10] =', request_cr[:10])
print('time_for_loop =', time_for_loop)

In [ ]:
start_time_vector = time.time()

A = df['status_code'] >= 400
B = df['is_slow'] & df['is_large']
mask = A | B
request_cr = df['request_id'].copy()[mask]

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

display(request_cr)

print('time_vector =', time_vector)

In [ ]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

#### Задание 3.1

In [ ]:
start_time_for_loop = time.time()

time_mobile_app_sum = 0
time_not_mobile_app_sum = 0
time_mobile_app_count = 0
time_not_mobile_app_count = 0

for i in range(len(df)):
    if df['client_type'][i] == 'mobile_app':
        time_mobile_app_sum = time_mobile_app_sum + df['response_time_ms'][i]
        time_mobile_app_count = time_mobile_app_count + 1
    else:
        time_not_mobile_app_sum = time_not_mobile_app_sum + df['response_time_ms'][i]
        time_not_mobile_app_count = time_not_mobile_app_count + 1
        
average_time_mobile_app = time_mobile_app_sum/time_mobile_app_count
average_time_not_mobile_app = time_not_mobile_app_sum/time_not_mobile_app_count

time_for_loop = time.time() - start_time_for_loop

print('average_time_mobile_app =', average_time_mobile_app)
print('average_time_not_mobile_app =', average_time_not_mobile_app)
print('time_for_loop =', time_for_loop)

In [ ]:
start_time_vector = time.time()

mask = df['client_type'] == 'mobile_app'
average_time_mobile_app = df['response_time_ms'][mask].mean()

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('average_time_mobile_app =', average_time_mobile_app)
print('time_vector =', time_vector)

In [ ]:
speed_up = round(time_for_loop / time_vector)

print('speed_up =', speed_up)

#### Задание 3.2

Цикл работает быстрее векторной операции!

In [ ]:
is_error = False

start_time_for_loop = time.time()

for i in range(len(df)):
    if (df['client_type'][i] == 'browser') and (df['status_code'][i] > 400):
        is_error = True
        break
        
time_for_loop = time.time() - start_time_for_loop
if time_for_loop == 0:
    time_for_loop = 0.0000005

print('is_error =', is_error)
print('time_for_loop =', time_for_loop)

In [ ]:
start_time_vector = time.time()

mask = (df['client_type'] == 'browser') & (df['status_code'] > 400)
is_error = mask.sum() > 0

time_vector = time.time() - start_time_vector
if time_vector == 0:
    time_vector = 0.0000005

print('is_error =', is_error)
print('time_vector =', time_vector)

Здесь мы вычисляем не ускорение, а замедление (поменяли местами числитель и знаменатель):

In [ ]:
speed_up = round(time_vector / time_for_loop)

print('speed_doun =', speed_up)

#### Комментарий к заданию 3.2

In [ ]:
mask

Мы использовали `break`, чтобы прервать цикл после первого выполнения условия `if`. Нам повезло. Это произошло на червертой итерации. При этом векторное построение булевой маски исполнило весь цикл: все 150 000 итераций. И хотя они выполнялись очень быстро, их было слишком много.